In [22]:
import os
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
from sklearn.metrics import mean_squared_error
from autogluon.tabular import TabularPredictor
import catboost as cb
from sklearn.neighbors import KNeighborsRegressor
import lightgbm as lgb
import xgboost as xgb
from funcs.train_model_funcs import clean_data, save_best_params, get_experiment_id, objective_autogluon, objective_catboost, objective_knn, objective_lgb, objective_xgb
import sys
import os
import pandas as pd
from tsfresh.feature_extraction import MinimalFCParameters, EfficientFCParameters
import sys
import mlflow
import dagshub
from funcs.train_model_funcs import clean_data, save_best_params, get_experiment_id, optimize_model, objective_autogluon, objective_catboost, objective_knn, objective_lgb, objective_xgb

target_feature = 'EXPGS'
feature_addition_rounds = 2
feature_dropping_threshold = 0.0002
fc_parameters = MinimalFCParameters()

# MLflow setup
mlflow.set_tracking_uri("https://dagshub.com/najibabounasr/MacroEconomicAPI.mlflow")
dagshub.init("MacroEconomicAPI", "najibabounasr", mlflow=True)
os.environ['MLFLOW_TRACKING_USERNAME'] = 'najibabounasr'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'fbaccfb8cf4e8d2d195cd05e9a53dbfe32323695'

# Load data
X_train = pd.read_csv('data/engineered/X_train_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
y_train = pd.read_csv('data/engineered/y_train_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
X_test = pd.read_csv('data/engineered/X_test_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
y_test = pd.read_csv('data/engineered/y_test_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)

# Clean data
X_train, y_train = clean_data(X_train, y_train)
X_test, y_test = clean_data(X_test, y_test)


Initialized MLflow to track repo "najibabounasr/MacroEconomicAPI"

Repository najibabounasr/MacroEconomicAPI initialized!

C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\1613183128.py:35: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  X_train = pd.read_csv('data/engineered/X_train_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\1613183128.py:36: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  y_train = pd.read_csv('data/engineered/y_train_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\161318312

# AutoGluon

In [42]:
import os
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
from sklearn.metrics import mean_squared_error, mean_absolute_error
from autogluon.tabular import TabularPredictor
from funcs.train_model_funcs import clean_data

def optimize_model(objective_function):
    study = optuna.create_study(direction='minimize')
    study.optimize(objective_function, n_trials=100)
    return study

def objective_autogluon(trial, X_train, y_train, X_test, y_test, target):
    param_grid = {
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'num_boost_round': trial.suggest_int('num_boost_round', 50, 100),
        'num_leaves': trial.suggest_int('num_leaves', 20, 50),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.5, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-4, 1e+1),
        'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-4, 1e+1),
        'verbose': -1
    }
    
    with mlflow.start_run(nested=True):
        train_data = pd.concat([X_train, y_train], axis=1)
        train_data.columns = list(X_train.columns) + [target]
        
        predictor = TabularPredictor(label=target, eval_metric='rmse').fit(
            train_data=train_data,
            hyperparameters={'GBM': param_grid},
            num_bag_folds=5,
            ag_args_fit={'num_gpus': 0, 'num_cpus': 1}
        )
        test_predictions = predictor.predict(X_test)
        train_predictions = predictor.predict(X_train)
        
        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))

        mlflow.log_metrics({
            'train_rmse': train_rmse,
            'test_rmse': test_rmse
        })
        
        for param_key, param_value in param_grid.items():
            mlflow.log_param(param_key, param_value)
        
        mlflow.set_tags({
            'target_feature': target,
            'feature_addition_rounds': feature_addition_rounds,
            'feature_dropping_threshold': feature_dropping_threshold,
            'fc_parameters': str(fc_parameters)
        })

    return test_rmse

def main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test):
    mlflow.set_experiment("AutoGluon Optimization")
    optimize_model(lambda trial: objective_autogluon(trial, X_train, y_train, X_test, y_test, target=target_feature))

if __name__ == "__main__":
    mlflow.set_tracking_uri("https://dagshub.com/najibabounasr/MacroEconomicAPI.mlflow")
    dagshub.init("MacroEconomicAPI", "najibabounasr", mlflow=True)
    os.environ['MLFLOW_TRACKING_USERNAME'] = 'najibabounasr'
    os.environ['MLFLOW_TRACKING_PASSWORD'] = 'fbaccfb8cf4e8d2d195cd05e9a53dbfe32323695'

    X_train = pd.read_csv('data/engineered/X_train_engineered.csv', index_col='Date', parse_dates=True)
    y_train = pd.read_csv('data/engineered/y_train_engineered.csv', index_col='Date', parse_dates=True)
    X_test = pd.read_csv('data/engineered/X_test_engineered.csv', index_col='Date', parse_dates=True)
    y_test = pd.read_csv('data/engineered/y_test_engineered.csv', index_col='Date', parse_dates=True)

    X_train, y_train = clean_data(X_train, y_train)
    X_test, y_test = clean_data(X_test, y_test)

    target_feature = 'EXPGS'
    feature_addition_rounds = 2
    feature_dropping_threshold = 0.0002
    fc_parameters = 'MinimalFCParameters'

    main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test)


Initialized MLflow to track repo "najibabounasr/MacroEconomicAPI"

Repository najibabounasr/MacroEconomicAPI initialized!

[I 2024-07-30 12:31:53,381] A new study created in memory with name: no-name-8e1e7233-beaf-49eb-83c3-741d9bc3c266
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\1443895155.py:18: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\1443895155.py:21: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\1443895155.py:22: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/o

# KNN

In [41]:
import os
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from funcs.train_model_funcs import clean_data

def optimize_model(objective_function):
    study = optuna.create_study(direction='minimize')
    study.optimize(objective_function, n_trials=100)
    return study

def validate_params(params):
    for key, value in params.items():
        if value is None:
            raise ValueError(f"Parameter '{key}' is None.")
    return params

def objective_knn(trial, X_train, y_train, X_test, y_test, target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters):
    param_grid = {
        'n_neighbors': trial.suggest_int('n_neighbors', 1, 50),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'algorithm': trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute']),
        'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        'p': trial.suggest_int('p', 1, 2)
    }
    
    try:
        param_grid = validate_params(param_grid)
        model = KNeighborsRegressor(**param_grid)
        model.fit(X_train, y_train)

        test_predictions = model.predict(X_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        train_predictions = model.predict(X_train)
        train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))

        with mlflow.start_run(nested=True):
            mlflow.log_metrics({
                'test_rmse': test_rmse,
                'train_rmse': train_rmse
            })
            for param_key, param_value in param_grid.items():
                mlflow.log_param(param_key, param_value)
            mlflow.set_tags({
                'target_feature': target_feature,
                'feature_addition_rounds': feature_addition_rounds,
                'feature_dropping_threshold': feature_dropping_threshold,
                'fc_parameters': str(fc_parameters)
            })

    except Exception as e:
        print(f"Error during model training or evaluation: {e}")
        return float('inf')  # Return a large value to indicate failure

    return test_rmse

def main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test):
    mlflow.set_experiment("KNN Optimization")
    optimize_model(lambda trial: objective_knn(trial, X_train, y_train, X_test, y_test, target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters))

if __name__ == "__main__":
    mlflow.set_tracking_uri("https://dagshub.com/najibabounasr/MacroEconomicAPI.mlflow")
    dagshub.init("MacroEconomicAPI", "najibabounasr", mlflow=True)
    os.environ['MLFLOW_TRACKING_USERNAME'] = 'najibabounasr'
    os.environ['MLFLOW_TRACKING_PASSWORD'] = 'fbaccfb8cf4e8d2d195cd05e9a53dbfe32323695'

    X_train = pd.read_csv('data/engineered/X_train_engineered.csv', index_col='Date', parse_dates=True)
    y_train = pd.read_csv('data/engineered/y_train_engineered.csv', index_col='Date', parse_dates=True)
    X_test = pd.read_csv('data/engineered/X_test_engineered.csv', index_col='Date', parse_dates=True)
    y_test = pd.read_csv('data/engineered/y_test_engineered.csv', index_col='Date', parse_dates=True)

    X_train, y_train = clean_data(X_train, y_train)
    X_test, y_test = clean_data(X_test, y_test)

    target_feature = 'EXPGS'
    feature_addition_rounds = 2
    feature_dropping_threshold = 0.0002
    fc_parameters = 'MinimalFCParameters'

    main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test)


Initialized MLflow to track repo "najibabounasr/MacroEconomicAPI"

Repository najibabounasr/MacroEconomicAPI initialized!

[I 2024-07-29 18:28:46,107] A new study created in memory with name: no-name-ed4ad382-ee02-475d-a8c6-964b96987e55
[I 2024-07-29 18:28:49,223] Trial 0 finished with value: 0.03714970916087207 and parameters: {'n_neighbors': 2, 'weights': 'distance', 'algorithm': 'brute', 'leaf_size': 44, 'p': 2}. Best is trial 0 with value: 0.03714970916087207.
[I 2024-07-29 18:28:52,279] Trial 1 finished with value: 0.03134371534720627 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'algorithm': 'kd_tree', 'leaf_size': 48, 'p': 2}. Best is trial 1 with value: 0.03134371534720627.
[I 2024-07-29 18:28:55,335] Trial 2 finished with value: 0.03044896001091866 and parameters: {'n_neighbors': 24, 'weights': 'distance', 'algorithm': 'auto', 'leaf_size': 50, 'p': 2}. Best is trial 2 with value: 0.03044896001091866.
[I 2024-07-29 18:28:58,601] Trial 3 finished with value: 0.03186808857668045 and parameters: {'n_neighbors': 10, 'weights': 'distance', 'algorithm': 'auto', 'leaf_size': 45, 'p': 2}. Best 

# CatBoost

In [43]:
import os
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
import catboost as cb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from funcs.train_model_funcs import clean_data

def optimize_model(objective_function):
    study = optuna.create_study(direction='minimize')
    study.optimize(objective_function, n_trials=100)
    return study

def objective_catboost(trial, X_train, y_train, X_test, y_test):
    param_grid = {
        'depth': trial.suggest_categorical('depth', [8, 12]),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'iterations': trial.suggest_int('iterations', 50, 200),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-4, 1e+1),
        'border_count': trial.suggest_categorical('border_count', [24, 26, 30]),
        'bagging_temperature': trial.suggest_uniform('bagging_temperature', 0, 10),
        'random_strength': trial.suggest_uniform('random_strength', 0, 1),
        'one_hot_max_size': trial.suggest_int('one_hot_max_size', 10, 50)
    }
    
    with mlflow.start_run(nested=True):
        model = cb.CatBoostRegressor(**param_grid, verbose=0)
        model.fit(X_train, y_train)

        test_predictions = model.predict(X_test)
        train_predictions = model.predict(X_train)

        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
        test_mse = mean_squared_error(y_test, test_predictions)
        train_mse = mean_squared_error(y_train, train_predictions)
        test_mae = mean_absolute_error(y_test, test_predictions)
        train_mae = mean_absolute_error(y_train, train_predictions)

        mlflow.log_metrics({
            'test_rmse': test_rmse,
            'train_rmse': train_rmse,
            'test_mse': test_mse,
            'train_mse': train_mse,
            'test_mae': test_mae,
            'train_mae': train_mae

        })
        
        for param_key, param_value in param_grid.items():
            mlflow.log_param(param_key, param_value)
        
        mlflow.set_tags({
            'target_feature': target_feature,
            'feature_addition_rounds': feature_addition_rounds,
            'feature_dropping_threshold': feature_dropping_threshold,
            'fc_parameters': str(fc_parameters)
        })

    return test_rmse

def main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test):
    mlflow.set_experiment("CatBoost Optimization")
    optimize_model(lambda trial: objective_catboost(trial, X_train, y_train, X_test, y_test))

if __name__ == "__main__":
    mlflow.set_tracking_uri("https://dagshub.com/najibabounasr/MacroEconomicAPI.mlflow")
    dagshub.init("MacroEconomicAPI", "najibabounasr", mlflow=True)
    os.environ['MLFLOW_TRACKING_USERNAME'] = 'najibabounasr'
    os.environ['MLFLOW_TRACKING_PASSWORD'] = 'fbaccfb8cf4e8d2d195cd05e9a53dbfe32323695'

    X_train = pd.read_csv('data/engineered/X_train_engineered.csv', index_col='Date', parse_dates=True)
    y_train = pd.read_csv('data/engineered/y_train_engineered.csv', index_col='Date', parse_dates=True)
    X_test = pd.read_csv('data/engineered/X_test_engineered.csv', index_col='Date', parse_dates=True)
    y_test = pd.read_csv('data/engineered/y_test_engineered.csv', index_col='Date', parse_dates=True)

    X_train, y_train = clean_data(X_train, y_train)
    X_test, y_test = clean_data(X_test, y_test)

    target_feature = 'EXPGS'
    feature_addition_rounds = 2
    feature_dropping_threshold = 0.0002
    fc_parameters = 'MinimalFCParameters'

    main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test)


Initialized MLflow to track repo "najibabounasr/MacroEconomicAPI"

Repository najibabounasr/MacroEconomicAPI initialized!

[I 2024-07-30 13:04:17,667] A new study created in memory with name: no-name-d190dec2-0807-469f-8795-5b53e078730f
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\334498134.py:19: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\334498134.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-4, 1e+1),
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\334498134.py:23: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://gi

# LightGBM

In [44]:
import os
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from funcs.train_model_funcs import clean_data

def optimize_model(objective_function):
    study = optuna.create_study(direction='minimize')
    study.optimize(objective_function, n_trials=100)
    return study

def objective_lgb(trial, X_train, y_train, X_test, y_test):
    param_grid = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 10, 100),
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart']),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.5, 1.0),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),
        'verbose': -1,
        'verbosity': -1
    }
    
    with mlflow.start_run(nested=True):
        model = lgb.LGBMRegressor(**param_grid)
        model.fit(X_train, y_train)

        test_predictions = model.predict(X_test)
        train_predictions = model.predict(X_train)


        test_predictions = model.predict(X_test)
        train_predictions = model.predict(X_train)

        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
        test_mse = mean_squared_error(y_test, test_predictions)
        train_mse = mean_squared_error(y_train, train_predictions)
        test_mae = mean_absolute_error(y_test, test_predictions)
        train_mae = mean_absolute_error(y_train, train_predictions)

        mlflow.log_metrics({
            'test_rmse': test_rmse,
            'train_rmse': train_rmse,
            'test_mse': test_mse,
            'train_mse': train_mse,
            'test_mae': test_mae,
            'train_mae': train_mae


        for param_key, param_value in param_grid.items():
            mlflow.log_param(param_key, param_value)
        
        mlflow.set_tags({
            'target_feature': target_feature,
            'feature_addition_rounds': feature_addition_rounds,
            'feature_dropping_threshold': feature_dropping_threshold,
            'fc_parameters': str(fc_parameters)
        })

    return test_rmse

def main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test):
    mlflow.set_experiment("LightGBM Optimization")
    optimize_model(lambda trial: objective_lgb(trial, X_train, y_train, X_test, y_test))

if __name__ == "__main__":
    mlflow.set_tracking_uri("https://dagshub.com/najibabounasr/MacroEconomicAPI.mlflow")
    dagshub.init("MacroEconomicAPI", "najibabounasr", mlflow=True)
    os.environ['MLFLOW_TRACKING_USERNAME'] = 'najibabounasr'
   


Initialized MLflow to track repo "najibabounasr/MacroEconomicAPI"

Repository najibabounasr/MacroEconomicAPI initialized!

# XGBoost

In [47]:
import os
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from funcs.train_model_funcs import clean_data, save_best_params, get_experiment_id

def objective_xgb(trial, X_train, y_train, X_test, y_test):
    mlflow.set_experiment("XGBoost Optimization")
    
    param_grid = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'lambda': trial.suggest_loguniform('lambda', 1e-4, 1e+1),
        'alpha': trial.suggest_loguniform('alpha', 1e-4, 1e+1)
    }
    
    with mlflow.start_run(nested=True):
        model = xgb.XGBRegressor(**param_grid)
        model.fit(X_train, y_train)

        test_predictions = model.predict(X_test)
        train_predictions = model.predict(X_train)

        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))
        test_mse = mean_squared_error(y_test, test_predictions)
        train_mse = mean_squared_error(y_train, train_predictions)
        test_mae = mean_absolute_error(y_test, test_predictions)
        train_mae = mean_absolute_error(y_train, train_predictions)

        mlflow.log_metrics({
            'test_rmse': test_rmse,
            'train_rmse': train_rmse,
            'test_mse': test_mse,
            'train_mse': train_mse,
            'test_mae': test_mae,
            'train_mae': train_mae})

        
        # Log model parameters
        for param_key, param_value in param_grid.items():
            mlflow.log_param(param_key, param_value)
        
        # Log tags
        mlflow.set_tags({
            'target_feature': target_feature,
            'feature_addition_rounds': feature_addition_rounds,
            'feature_dropping_threshold': feature_dropping_threshold,
            'fc_parameters': str(fc_parameters)
        })

target_feature = 'EXPGS'
feature_addition_rounds = 2
feature_dropping_threshold = 0.0002
fc_parameters = MinimalFCParameters()

# MLflow setup
mlflow.set_tracking_uri("https://dagshub.com/najibabounasr/MacroEconomicAPI.mlflow")
dagshub.init("MacroEconomicAPI", "najibabounasr", mlflow=True)
os.environ['MLFLOW_TRACKING_USERNAME'] = 'najibabounasr'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'fbaccfb8cf4e8d2d195cd05e9a53dbfe32323695'

# Load data
X_train = pd.read_csv('data/engineered/X_train_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
y_train = pd.read_csv('data/engineered/y_train_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
X_test = pd.read_csv('data/engineered/X_test_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
y_test = pd.read_csv('data/engineered/y_test_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)

# Clean data
X_train, y_train = clean_data(X_train, y_train)
X_test, y_test = clean_data(X_test, y_test)

def main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test):
    print(f"Target feature: {target_feature}")
    print(f"Feature addition rounds: {feature_addition_rounds}")
    print(f"Feature dropping threshold: {feature_dropping_threshold}")
    print(f"Feature engineering parameters: {fc_parameters}")
    
    # Optimize XGBoost model
    optimize_model(lambda trial: objective_xgb(trial, X_train, y_train, X_test, y_test))

if __name__ == "__main__":
    main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test)


Initialized MLflow to track repo "najibabounasr/MacroEconomicAPI"

Repository najibabounasr/MacroEconomicAPI initialized!

C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\692464071.py:71: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  X_train = pd.read_csv('data/engineered/X_train_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\692464071.py:72: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  y_train = pd.read_csv('data/engineered/y_train_engineered.csv', index_col='Date', parse_dates=True, infer_datetime_format=True)
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\692464071.p

Target feature: EXPGS
Feature addition rounds: 2
Feature dropping threshold: 0.0002
Feature engineering parameters: {'sum_values': None, 'median': None, 'mean': None, 'length': None, 'standard_deviation': None, 'variance': None, 'root_mean_square': None, 'maximum': None, 'absolute_maximum': None, 'minimum': None}


C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\692464071.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\692464071.py:18: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
C:\Users\nabounaser\AppData\Local\Temp\ipykernel_26036\692464071.py:19: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1

# REVISED AUTOGLUON